In [1]:
# -----------------------------------------------
# One-cell SVM experiment
# RBF kernel, class_weight='balanced'
# Compared against frozen MLP-1 baseline
# -----------------------------------------------
import numpy as np
import os
from sklearn.svm import SVC
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
os.chdir("..")

def optimal_threshold_f1(
    y_true, y_scores,
    thresholds=np.arange(0.01, 0.99, 0.01),
):
    best_f1, best_t = 0, 0.5
    for t in thresholds:
        preds = (y_scores >= t).astype(int)
        f1    = f1_score(y_true, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1

# Load frozen embeddings
X_train = np.load("data/processed/embeddings/X_train.npy")
y_train = np.load("data/processed/embeddings/y_train.npy")
X_cal   = np.load("data/processed/embeddings/X_cal.npy")
y_cal   = np.load("data/processed/embeddings/y_cal.npy")
X_test  = np.load("data/processed/embeddings/X_test.npy")
y_test  = np.load("data/processed/embeddings/y_test.npy")

print("Training SVM (RBF kernel)...")
svm = SVC(
    kernel='rbf',
    class_weight='balanced',
    probability=True,      # needed for threshold optimization
    C=1.0,
    gamma='scale',         # 1 / (n_features * X.var()) — sensible default
    random_state=42,
)
svm.fit(X_train, y_train)
print("Done.")

# Threshold optimization on cal set
cal_scores  = svm.predict_proba(X_cal)[:, 1]
t, cal_f1   = optimal_threshold_f1(y_cal, cal_scores)

# Evaluate on test set
test_scores = svm.predict_proba(X_test)[:, 1]
y_pred      = (test_scores >= t).astype(int)

prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)
f1   = f1_score(y_test, y_pred, zero_division=0)

print(f"\n=== SVM RBF (threshold={t:.2f}) ===")
print(classification_report(y_test, y_pred, digits=3))

print("=" * 55)
print(f"{'Classifier':<25} {'P':>8} {'R':>8} {'F1':>8}")
print("-" * 55)
print(f"{'Frozen LR':<25} {'0.085':>8} {'0.729':>8} {'0.152':>8}")
print(f"{'Frozen MLP-1':<25} {'0.190':>8} {'0.464':>8} {'0.270':>8}")
print(f"{'Frozen MLP-1+CoT':<25} {'0.223':>8} {'0.422':>8} {'0.292':>8}")
print(f"{'SVM RBF':<25} {prec:>8.3f} {rec:>8.3f} {f1:>8.3f}")
print("=" * 55)

Training SVM (RBF kernel)...
Done.

=== SVM RBF (threshold=0.10) ===
              precision    recall  f1-score   support

           0      0.994     0.989     0.991     19669
           1      0.241     0.359     0.289       192

    accuracy                          0.983     19861
   macro avg      0.617     0.674     0.640     19861
weighted avg      0.986     0.983     0.985     19861

Classifier                       P        R       F1
-------------------------------------------------------
Frozen LR                    0.085    0.729    0.152
Frozen MLP-1                 0.190    0.464    0.270
Frozen MLP-1+CoT             0.223    0.422    0.292
SVM RBF                      0.241    0.359    0.289


In [2]:
# Build augmented sets — reuse filtered arrays from earlier
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer('all-MiniLM-L6-v2')
X_real_pos = X_train[y_train == 1]

# CoT positives
df_cot    = pd.read_parquet("data/synthetic/positive_cot_70b_raw.parquet")
X_cot     = encoder.encode(df_cot['sentence'].tolist(),
                            convert_to_numpy=True, show_progress_bar=False)
sim_cot   = cosine_similarity(X_cot, X_real_pos).max(axis=1)
mask_cot  = sim_cot >= 0.6
X_cot_f   = X_cot[mask_cot]
y_cot_f   = np.ones(mask_cot.sum(), dtype=int)
print(f"CoT survivors: {mask_cot.sum()}")

# Contrastive negatives
df_cont   = pd.read_parquet("data/synthetic/hard_contrastive_70b_raw.parquet")
X_cont    = encoder.encode(df_cont['sentence'].tolist(),
                            convert_to_numpy=True, show_progress_bar=False)
sim_cont  = cosine_similarity(X_cont, X_real_pos).max(axis=1)
mask_cont = (sim_cont >= 0.4) & (sim_cont <= 0.9)
X_cont_f  = X_cont[mask_cont]
y_cont_f  = np.zeros(mask_cont.sum(), dtype=int)
print(f"Contrastive survivors: {mask_cont.sum()}")

# Augmented sets
X_train_cot  = np.vstack([X_train, X_cot_f])
y_train_cot  = np.concatenate([y_train, y_cot_f])

X_train_cont = np.vstack([X_train, X_cont_f])
y_train_cont = np.concatenate([y_train, y_cont_f])

del encoder
print("Augmented sets ready.")

c:\Users\vkamat01\hedging-txtclf-experiments\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 777.86it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CoT survivors: 608
Contrastive survivors: 651
Augmented sets ready.


In [3]:
results_svm = {}

for name, X_tr, y_tr in [
    ('SVM RBF + CoT',         X_train_cot,  y_train_cot),
    ('SVM RBF + Contrastive', X_train_cont, y_train_cont),
]:
    print(f"\nTraining {name} "
          f"(n={len(X_tr):,}, pos={y_tr.sum()})...")
    svm = SVC(
        kernel='rbf',
        class_weight='balanced',
        probability=True,
        C=1.0,
        gamma='scale',
        random_state=42,
    )
    svm.fit(X_tr, y_tr)

    cal_scores  = svm.predict_proba(X_cal)[:, 1]
    t, _        = optimal_threshold_f1(y_cal, cal_scores)
    test_scores = svm.predict_proba(X_test)[:, 1]
    y_pred      = (test_scores >= t).astype(int)

    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)

    print(classification_report(y_test, y_pred, digits=3))
    results_svm[name] = {'precision': prec, 'recall': rec,
                         'f1': f1, 'threshold': t}

# Summary
print("=" * 55)
print(f"{'Classifier':<25} {'P':>8} {'R':>8} {'F1':>8}")
print("-" * 55)
print(f"{'SVM RBF (real)':<25} {'0.241':>8} {'0.359':>8} {'0.289':>8}")
for name, r in results_svm.items():
    print(f"{name:<25} {r['precision']:>8.3f} "
          f"{r['recall']:>8.3f} {r['f1']:>8.3f}")
print(f"{'Frozen MLP-1+CoT':<25} {'0.223':>8} {'0.422':>8} {'0.292':>8}")
print("=" * 55)


Training SVM RBF + CoT (n=70,118, pos=1282)...
              precision    recall  f1-score   support

           0      0.993     0.994     0.993     19669
           1      0.290     0.234     0.259       192

    accuracy                          0.987     19861
   macro avg      0.641     0.614     0.626     19861
weighted avg      0.986     0.987     0.986     19861


Training SVM RBF + Contrastive (n=70,161, pos=674)...
              precision    recall  f1-score   support

           0      0.993     0.990     0.992     19669
           1      0.251     0.328     0.284       192

    accuracy                          0.984     19861
   macro avg      0.622     0.659     0.638     19861
weighted avg      0.986     0.984     0.985     19861

Classifier                       P        R       F1
-------------------------------------------------------
SVM RBF (real)               0.241    0.359    0.289
SVM RBF + CoT                0.290    0.234    0.259
SVM RBF + Contrastive       